# NASA PSI data builds in Colab

Build PSI-25 BASS-II and PSI-20 ACME BRE from their public NASA PSI records. PSI-20 outputs preserve the source's daily-summary granularity.

Run from top to bottom. The Colab steps clone the repository into runtime storage, save build logs and optional video probe results to the mounted team folder, and copy only the small app JSON bundles to the repository checkout. Review the generated files before committing them.

Sources: [PSI-25](https://doi.org/10.60555/4qc4-de67) · [PSI-20](https://doi.org/10.60555/vb93-3y72)

In [1]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

REPOSITORY = "https://github.com/eventhorizontr/Dashboardv1.git"
REF = os.environ.get("PSI_PIPELINE_REF", "main")
WORKSPACE = Path(os.environ.get("PSI_PIPELINE_WORKSPACE", "/content/fire-in-freefall"))
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PROBE_VIDEOS = False

In [2]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
repository = WORKSPACE / "Dashboardv1"
if not repository.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPOSITORY, str(repository)], check=True)
else:
    subprocess.run(["git", "-C", str(repository), "rev-parse", "--verify", "HEAD"], check=True)

for name in ("build_psi25.py", "build_psi20.py"):
    if not (repository / "pipeline" / name).is_file():
        raise FileNotFoundError(f"{name} is missing in this checkout. Use a ref that contains #17.")
revision = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
print(f"Pipeline revision {revision[:12]}")

Pipeline revision 6a47dbc5f250


In [3]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive/Nasa Space Apps Data 2026")
else:
    drive_root = Path(os.environ.get("PSI_PIPELINE_STORAGE", str(WORKSPACE / "local-storage")))

intermediates = drive_root / "pipeline-runs" / revision[:12]
intermediates.mkdir(parents=True, exist_ok=True)
staging = WORKSPACE / "staging"

def run_pipeline(label, script, *arguments):
    log_path = intermediates / f"{label}.log"
    with log_path.open("w", encoding="utf8") as log:
        result = subprocess.run(
            [sys.executable, str(repository / "pipeline" / script), *map(str, arguments)],
            stdout=log, stderr=subprocess.STDOUT,
        )
    if result.returncode:
        raise RuntimeError(f"{label} failed. See {log_path}")
    print(log_path.read_text(encoding="utf8").strip())

In [4]:
for number in (25, 20):
    run_pipeline(f"build-psi{number}", f"build_psi{number}.py", staging)

if PROBE_VIDEOS:
    run_pipeline(
        "probe-psi25", "psi25_video.py", "probe", staging,
        "--cache", intermediates / "psi25-video-probes.json",
    )

files=695 experiments=129 with_video=123 qa_flagged=13 unmatched_video_ids=[]


PSI-20: 18 daily summaries, 2361 cataloged files


In [5]:
def promote_outputs(accession):
    outputs = staging / "public" / "data" / accession
    names = ("manifest.json", "experiments.json")
    manifest, dataset = [json.loads((outputs / name).read_text(encoding="utf8")) for name in names]
    if manifest["accession"] != accession.upper() or dataset["accession"] != accession.upper():
        raise ValueError(f"Incorrect accession in {outputs}")
    if dataset["count"] != len(dataset["experiments"]):
        raise ValueError(f"Experiment count mismatch in {outputs}")
    file_count = sum(len(group) for group in manifest["groups"].values())
    if manifest["totals"]["files"] != file_count:
        raise ValueError(f"Manifest count mismatch in {outputs}")
    if any((outputs / name).stat().st_size >= 5_000_000 for name in names):
        raise ValueError(f"Found a non-small JSON bundle in {outputs}")

    destination = repository / "public" / "data" / accession
    destination.mkdir(parents=True, exist_ok=True)
    for name in names:
        shutil.copy2(outputs / name, destination / name)
    print(f"{accession.upper()}: {dataset['count']} records and {file_count} files")

for accession in ("psi-25", "psi-20"):
    promote_outputs(accession)

PSI-25: 129 records and 695 files
PSI-20: 18 records and 2361 files


### Review and handoff

Inspect the four generated JSON files in the repository checkout. Run the repository's `pnpm test` contract checks before opening a PR. Heavy probe intermediates and logs stay in the shared Drive folder. The app consumes only the promoted JSON files.